In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
subway_final = pd.read_csv("../../data/processed/team/01_passenger.csv")
subway_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   지하철역     249 non-null    object 
 1   호선명      249 non-null    object 
 2   총승차인원    249 non-null    float64
 3   무임승차인원   249 non-null    float64
 4   총하차인원    249 non-null    float64
 5   무임하차인원   249 non-null    float64
 6   전체승하차    249 non-null    float64
 7   무임승하차    249 non-null    float64
 8   무임승하차비중  249 non-null    float64
dtypes: float64(7), object(2)
memory usage: 17.6+ KB


In [4]:
master_df = pd.read_csv("../../data/processed/team/02_facility.csv")
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3385 entries, 0 to 3384
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   운영노선명   574 non-null    object
 1   역명      3385 non-null   object
 2   승강기명    3385 non-null   object
 3   승강기 구분  3385 non-null   object
dtypes: object(4)
memory usage: 105.9+ KB


In [84]:
master_df.rename(columns = {'운영노선명' : '호선명', '역명' :'지하철역'}, inplace=True)
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3385 entries, 0 to 3384
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   호선명     574 non-null    object
 1   지하철역    3385 non-null   object
 2   승강기명    3385 non-null   object
 3   승강기 구분  3385 non-null   object
dtypes: object(4)
memory usage: 105.9+ KB


In [85]:
line_map = subway_final.set_index('지하철역')['호선명']
line_map

지하철역
제기동       1
종각        1
종로5가      1
청량리       1
강남        2
       ... 
청구      5,6
충무로     3,4
충정로     2,5
태릉입구    6,7
합정      2,6
Name: 호선명, Length: 249, dtype: object

In [86]:
master_df['호선명'] = master_df['호선명'].replace('', np.nan)
master_df['호선명'] = master_df['호선명'].fillna(master_df['지하철역'].map(line_map))

In [87]:
master_df.loc[master_df['호선명'] == '9호선', '호선명'] = "9"
master_df[master_df['호선명'].isna()]

,호선명,지하철역,승강기명,승강기 구분
578,NaN,서울역(1),승강기)에스컬레이터-서울(1)역 4번출구 4호기,ES
579,NaN,서울역(1),승강기)에스컬레이터-서울(1)역 4번출구 5호기,ES
580,NaN,서울역(1),승강기)에스컬레이터-서울(1)역 상행(9-3) 3호기,ES
581,NaN,서울역(1),승강기)에스컬레이터-서울(1)역 하행(2-3) 2호기,ES
582,NaN,서울역(1),승강기)에스컬레이터-서울역(1) 4호선연결통로 1호기,ES
...,...,...,...,...
3380,NaN,남위례,승강기)에스컬레이터-남위례 내부 5호기,ES
3381,NaN,남위례,승강기)에스컬레이터-남위례 내부 6호기,ES
3382,NaN,남위례,승강기)엘리베이터-남위례 내부 1호기,EV
3383,NaN,남위례,승강기)엘리베이터-남위례 내부 2호기,EV


In [88]:
master_df['호선명'] = master_df['호선명'].fillna(master_df['지하철역'].str.split("(").str[1].str.replace(")", ""))
master_df['호선명'].unique()

array(['9', '7', '1', '2', '3', '4', '전쟁기념관', '6', '5', nan, '뚝섬한강공원',
       '8'], dtype=object)

In [89]:
master_df.loc[master_df['호선명'] == '전쟁기념관', '호선명'] = "4"
master_df.loc[master_df['호선명'] == '뚝섬한강공원', '호선명'] = "7"
master_df['호선명'].unique()

array(['9', '7', '1', '2', '3', '4', '6', '5', nan, '8'], dtype=object)

In [90]:
master_df['지하철역'] = master_df['지하철역'].str.split("(").str[0]
master_df

,호선명,지하철역,승강기명,승강기 구분
0,9,개화,(1F) 하선승강장 6-4 근처\n(2F) 안전관리실 옆 1호기,EV
1,9,개화,(1F) 2번출구 옆\n(2F) 편의점 앞 2호기,EV
2,9,김포공항,(B1) 안전관리실 옆\n(B3) 상부본선 승강장 4-2 지점 2호기,EV
3,9,김포공항,(B1) 안전관리실 앞\n(B4) 하선 승강장 1-1 지점 3호기,EV
4,9,김포공항,(B1) 안전관리실 앞\n(B4) 하선 승강장 1-1 지점 4호기,EV
...,...,...,...,...
3380,NaN,남위례,승강기)에스컬레이터-남위례 내부 5호기,ES
3381,NaN,남위례,승강기)에스컬레이터-남위례 내부 6호기,ES
3382,NaN,남위례,승강기)엘리베이터-남위례 내부 1호기,EV
3383,NaN,남위례,승강기)엘리베이터-남위례 내부 2호기,EV


In [91]:
# 서울 시내에 있는 역이 아닌 행 삭제
non_seoul = [
    '미사', '하남검단산', '하남시청', '하남풍산',
    '광명사거리', '굴포천', '까치울', '부천시청', '부평구청',
    '삼산체육관', '상동', '신중동', '춘의', '철산', 
    '남위례', '남한산성입구', '단대오거리',
    '모란', '복정', '산성', '수진', '신흥'
]
master_df = master_df[~master_df['지하철역'].isin(non_seoul)]

In [92]:
master_df[~master_df['호선명'].isin(["1", "2", "3", "4", "5", "6", "7", "8", "9"])]['호선명'].unique()

array([], dtype=object)

In [93]:
# '지하철역' 과 '승강기 구분' 기준으로 개수 집계하기
master_pivot = master_df.pivot_table(
                                index=['지하철역'],
                                columns='승강기 구분',
                                aggfunc='size',
                                fill_value=0
                                ).reset_index() 

In [94]:
# subway_final과 master_df 병합하기
merged = pd.merge(subway_final, master_pivot, on='지하철역', how='left')
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,MW,WL
0,제기동,1,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25,2.0,3.0,0.0,0.0
1,종각,1,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33,2.0,4.0,0.0,0.0
2,종로5가,1,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61,0.0,3.0,0.0,0.0
3,청량리,1,662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58,6.0,2.0,0.0,1.0
4,강남,2,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64,0.0,4.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
244,청구,"5,6",219899.0,44565.0,211920.0,42532.0,431819.0,87097.0,20.17,10.0,5.0,0.0,0.0
245,충무로,"3,4",867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73,23.0,4.0,0.0,0.0
246,충정로,"2,5",417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96,8.0,4.0,0.0,0.0
247,태릉입구,"6,7",438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57,24.0,6.0,0.0,0.0


In [95]:
merged.loc[merged['EV'] == 0, '지하철역']

46     도곡
201    사평
Name: 지하철역, dtype: object

In [96]:
# 엘리베이터 수 조정하기
merged.loc[merged['지하철역']=='도곡', 'EV'] = 2
merged.loc[merged['지하철역']=='사평', 'EV'] = 4

In [97]:
# 개명된 역 삭제하기
drop_stations = ['뚝섬유원지', '당고개', '신내']
merged = merged[~merged['지하철역'].isin(drop_stations)]

In [98]:
# ES + EV 편의시설의 개수를 집계하는 'TOTAL' 컬럼 생성하기
merged['TOTAL'] = merged['ES'] + merged['EV']
merged[['ES', 'EV', 'TOTAL']] = merged[['ES', 'EV', 'TOTAL']].astype('Int64')
merged

/var/folders/rx/ylxcpmnn5hz6xwbkxhnfnxfm0000gn/T/ipykernel_4122/1836546633.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged['TOTAL'] = merged['ES'] + merged['EV']
/var/folders/rx/ylxcpmnn5hz6xwbkxhnfnxfm0000gn/T/ipykernel_4122/1836546633.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged[['ES', 'EV', 'TOTAL']] = merged[['ES', 'EV', 'TOTAL']].astype('Int64')


,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,MW,WL,TOTAL
0,제기동,1,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25,2,3,0.0,0.0,5
1,종각,1,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33,2,4,0.0,0.0,6
2,종로5가,1,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61,0,3,0.0,0.0,3
3,청량리,1,662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58,6,2,0.0,1.0,8
4,강남,2,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64,0,4,0.0,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244,청구,"5,6",219899.0,44565.0,211920.0,42532.0,431819.0,87097.0,20.17,10,5,0.0,0.0,15
245,충무로,"3,4",867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73,23,4,0.0,0.0,27
246,충정로,"2,5",417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96,8,4,0.0,0.0,12
247,태릉입구,"6,7",438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57,24,6,0.0,0.0,30


In [99]:
merged = merged.drop(['MW', 'WL'], axis=1)
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
0,제기동,1,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25,2,3,5
1,종각,1,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33,2,4,6
2,종로5가,1,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61,0,3,3
3,청량리,1,662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58,6,2,8
4,강남,2,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64,0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
244,청구,"5,6",219899.0,44565.0,211920.0,42532.0,431819.0,87097.0,20.17,10,5,15
245,충무로,"3,4",867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73,23,4,27
246,충정로,"2,5",417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96,8,4,12
247,태릉입구,"6,7",438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57,24,6,30


In [100]:
# 서울역 이름 서울로 변경
merged['지하철역'] = merged['지하철역'].replace('서울역', '서울')

In [101]:
merged.shape

(246, 12)

In [102]:
merged[merged['지하철역'].isin(['이수', '총신대입구'])]

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
77,총신대입구,4,564012.0,159811.0,610462.0,167328.0,1174474.0,327139.0,27.85,4,4,8
173,이수,7,479384.0,85118.0,463584.0,80754.0,942968.0,165872.0,17.59,23,3,26


In [103]:
# 이수역과 총신대입구역을 합치는 코드
isu_df = merged[merged['지하철역'].isin(['이수', '총신대입구'])]

new_isu_row = {
    '지하철역': '이수',
    '호선명': '4,7',
    '총승차인원': isu_df['총승차인원'].sum(),
    '무임승차인원': isu_df['무임승차인원'].sum(),
    '총하차인원': isu_df['총하차인원'].sum(),
    '무임하차인원': isu_df['무임하차인원'].sum(),
    '전체승하차': isu_df['전체승하차'].sum(),
    '무임승하차': isu_df['무임승하차'].sum(),
    'ES': isu_df['ES'].sum(),
    'EV': isu_df['EV'].sum(),
    'TOTAL': isu_df['TOTAL'].sum()
}

new_isu_row['무임승하차비중'] = round((new_isu_row['무임승하차'] / new_isu_row['전체승하차']) * 100, 2)

merged = merged[~merged['지하철역'].isin(['이수', '총신대입구'])].copy()
merged = pd.concat([merged, pd.DataFrame([new_isu_row])], ignore_index=True)

merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
0,제기동,1,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25,2,3,5
1,종각,1,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33,2,4,6
2,종로5가,1,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61,0,3,3
3,청량리,1,662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58,6,2,8
4,강남,2,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64,0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
240,충무로,"3,4",867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73,23,4,27
241,충정로,"2,5",417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96,8,4,12
242,태릉입구,"6,7",438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57,24,6,30
243,합정,"2,6",1379809.0,113748.0,1461301.0,111461.0,2841110.0,225209.0,7.93,16,7,23


In [104]:
# 인원수 정수화
target_cols = ['총승차인원', '무임승차인원', '총하차인원', '무임하차인원', '전체승하차', '무임승하차']
merged[target_cols] = merged[target_cols].astype(int)
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
0,제기동,1,501317,269661,511782,290055,1013099,559716,55.25,2,3,5
1,종각,1,1112527,151760,1083408,140929,2195935,292689,13.33,2,4,6
2,종로5가,1,712905,247967,697910,240352,1410815,488319,34.61,0,3,3
3,청량리,1,662997,280334,656727,281655,1319724,561989,42.58,6,2,8
4,강남,2,2302759,161221,2246564,140970,4549323,302191,6.64,0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
240,충무로,"3,4",867520,119883,893846,121889,1761366,241772,13.73,23,4,27
241,충정로,"2,5",417744,63616,436275,64124,854019,127740,14.96,8,4,12
242,태릉입구,"6,7",438855,87568,443702,85169,882557,172737,19.57,24,6,30
243,합정,"2,6",1379809,113748,1461301,111461,2841110,225209,7.93,16,7,23


In [105]:
# 지축역은 서울시 경계와 애매하기 때문에 삭제
merged = merged[~(merged['지하철역'] == '지축')]
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
0,제기동,1,501317,269661,511782,290055,1013099,559716,55.25,2,3,5
1,종각,1,1112527,151760,1083408,140929,2195935,292689,13.33,2,4,6
2,종로5가,1,712905,247967,697910,240352,1410815,488319,34.61,0,3,3
3,청량리,1,662997,280334,656727,281655,1319724,561989,42.58,6,2,8
4,강남,2,2302759,161221,2246564,140970,4549323,302191,6.64,0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
240,충무로,"3,4",867520,119883,893846,121889,1761366,241772,13.73,23,4,27
241,충정로,"2,5",417744,63616,436275,64124,854019,127740,14.96,8,4,12
242,태릉입구,"6,7",438855,87568,443702,85169,882557,172737,19.57,24,6,30
243,합정,"2,6",1379809,113748,1461301,111461,2841110,225209,7.93,16,7,23


In [106]:
# 가중치를 적용하여 시설부하지수 컬럼 생성
merged = merged[['지하철역', '호선명', '총승차인원', '무임승차인원', '총하차인원', '무임하차인원', '전체승하차', '무임승하차', '무임승하차비중', 'EV', 'ES', 'TOTAL']].copy()
merged['시설부하지수'] = merged['무임승하차'] / (merged['EV'] * 4 + merged['ES'] * 1)
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,EV,ES,TOTAL,시설부하지수
0,제기동,1,501317,269661,511782,290055,1013099,559716,55.25,3,2,5,39979.714286
1,종각,1,1112527,151760,1083408,140929,2195935,292689,13.33,4,2,6,16260.5
2,종로5가,1,712905,247967,697910,240352,1410815,488319,34.61,3,0,3,40693.25
3,청량리,1,662997,280334,656727,281655,1319724,561989,42.58,2,6,8,40142.071429
4,강남,2,2302759,161221,2246564,140970,4549323,302191,6.64,4,0,4,18886.9375
...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,충무로,"3,4",867520,119883,893846,121889,1761366,241772,13.73,4,23,27,6199.282051
241,충정로,"2,5",417744,63616,436275,64124,854019,127740,14.96,4,8,12,5322.5
242,태릉입구,"6,7",438855,87568,443702,85169,882557,172737,19.57,6,24,30,3598.6875
243,합정,"2,6",1379809,113748,1461301,111461,2841110,225209,7.93,7,16,23,5118.386364


In [122]:
# 시설부하지수 소수점 둘째까리에서 반올림
merged['시설부하지수'] = round(merged['시설부하지수'], 2)
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,EV,ES,TOTAL,시설부하지수
0,제기동,1,501317,269661,511782,290055,1013099,559716,55.25,3,2,5,39979.71
1,종각,1,1112527,151760,1083408,140929,2195935,292689,13.33,4,2,6,16260.5
2,종로5가,1,712905,247967,697910,240352,1410815,488319,34.61,3,0,3,40693.25
3,청량리,1,662997,280334,656727,281655,1319724,561989,42.58,2,6,8,40142.07
4,강남,2,2302759,161221,2246564,140970,4549323,302191,6.64,4,0,4,18886.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,충무로,"3,4",867520,119883,893846,121889,1761366,241772,13.73,4,23,27,6199.28
241,충정로,"2,5",417744,63616,436275,64124,854019,127740,14.96,4,8,12,5322.5
242,태릉입구,"6,7",438855,87568,443702,85169,882557,172737,19.57,6,24,30,3598.69
243,합정,"2,6",1379809,113748,1461301,111461,2841110,225209,7.93,7,16,23,5118.39


In [125]:
# merged.to_excel("subway_merged_base.xlsx", index=False)

In [126]:
# merged.to_csv("subway_merged_base.csv", index=False)